# Similarity and Search

**Goal:** Learn how to compare text using cosine similarity and build a mini semantic search engine.

**Prereqs:** Complete [00_tokenization_and_embeddings.ipynb](./00_tokenization_and_embeddings.ipynb) first.

**Libraries:** `sentence-transformers`, `scikit-learn`, `numpy`

## The Foundation of RAG

**Go/TS comparison:** In your Go RAG project, you sent text to an LLM and got answers back. But how did the system know *which* documents to send? That's the retrieval step — and it works by computing similarity between the query embedding and all document embeddings. This notebook builds that from scratch.

## Cosine Similarity from Scratch

**Go/TS comparison:** In Go, you'd compare strings with `==` or maybe Levenshtein distance. Cosine similarity compares *vectors* — it measures the angle between them, ignoring magnitude. Two vectors pointing in the same direction score 1.0, perpendicular score 0.0, opposite score -1.0.

In [ ]:
import numpy as np

def cosine_similarity_manual(vec_a, vec_b):
    """Compute cosine similarity from scratch — no libraries"""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = sum(x ** 2 for x in vec_a) ** 0.5
    magnitude_b = sum(x ** 2 for x in vec_b) ** 0.5
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)

# Test with simple vectors
a = [1, 0, 0]
b = [1, 0, 0]  # Same direction
c = [0, 1, 0]  # Perpendicular
d = [-1, 0, 0] # Opposite

print(f"Same direction:  {cosine_similarity_manual(a, b):.4f}")  # 1.0
print(f"Perpendicular:   {cosine_similarity_manual(a, c):.4f}")  # 0.0
print(f"Opposite:        {cosine_similarity_manual(a, d):.4f}")  # -1.0

In [ ]:
# Verify against sklearn
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# sklearn expects 2D arrays
a_2d = np.array([[1, 2, 3]])
b_2d = np.array([[4, 5, 6]])

manual = cosine_similarity_manual(a_2d[0], b_2d[0])
sklearn = sklearn_cosine(a_2d, b_2d)[0][0]

print(f"Manual:  {manual:.6f}")
print(f"sklearn: {sklearn:.6f}")
print(f"Match:   {abs(manual - sklearn) < 1e-10}")

### ✍️ In Your Own Words

Why does cosine similarity ignore vector magnitude? When would this be an advantage? Write your answer here.

## Semantic vs Lexical Similarity

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

# Pairs to compare
pairs = [
    ("The cat sat on the mat", "A feline rested on the rug"),       # Same meaning, different words
    ("The cat sat on the mat", "The cat sat on the mat"),           # Identical
    ("bank of the river", "bank account balance"),                   # Same word, different meaning
    ("I love programming", "Coding is my passion"),                  # Semantic match
    ("The weather is nice", "Python is a programming language"),     # Unrelated
]

for s1, s2 in pairs:
    emb = model.encode([s1, s2])
    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    print(f"{sim:.3f}  '{s1}' ↔ '{s2}'")

In [ ]:
# Experiment: lexical overlap doesn't mean semantic similarity
tricky_pairs = [
    ("The dog bit the man", "The man bit the dog"),                 # Same words, different meaning
    ("He is not happy", "He is happy"),                              # Negation
    ("The movie was not bad", "The movie was good"),                 # Double negative ≈ positive
]

for s1, s2 in tricky_pairs:
    emb = model.encode([s1, s2])
    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    print(f"{sim:.3f}  '{s1}' ↔ '{s2}'")

print("\nNote: models often struggle with negation — a known limitation")

### ✍️ In Your Own Words

Give an example where cosine similarity on embeddings would give a better result than simple string matching. Write your answer here.

## Building a Mini Search Engine

**Go/TS comparison:** This is literally what a vector database (ChromaDB, Pinecone, Weaviate) does under the hood. You're building the core retrieval algorithm that powers RAG — embed everything, embed the query, find the closest matches.

In [ ]:
# Our "document database"
documents = [
    "Python is a popular programming language for data science",
    "JavaScript is widely used for web development",
    "Machine learning models need large amounts of training data",
    "Docker containers package applications with their dependencies",
    "Neural networks are inspired by biological brain structures",
    "REST APIs use HTTP methods like GET, POST, PUT, DELETE",
    "Vector databases store embeddings for similarity search",
    "Go is known for its concurrency model using goroutines",
    "Natural language processing converts text to numerical representations",
    "Kubernetes orchestrates containerized applications at scale",
]

# Embed all documents (do this once, store the results)
doc_embeddings = model.encode(documents)
print(f"Embedded {len(documents)} documents")
print(f"Embedding matrix shape: {doc_embeddings.shape}")

In [ ]:
def search(query, documents, doc_embeddings, model, top_k=3):
    """Semantic search — embed query, find closest documents"""
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Rank by similarity
    ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    print(f"Query: '{query}'\n")
    for rank, (idx, score) in enumerate(ranked[:top_k], 1):
        print(f"  {rank}. [{score:.3f}] {documents[idx]}")
    print()

# Try different queries
search("How do I build a web app?", documents, doc_embeddings, model)
search("What is AI?", documents, doc_embeddings, model)
search("container orchestration", documents, doc_embeddings, model)

In [ ]:
# Experiment: the query doesn't need to match any document words
search("fast concurrent server language", documents, doc_embeddings, model)
search("turning human language into numbers", documents, doc_embeddings, model)
search("packaging software for deployment", documents, doc_embeddings, model)

### ✍️ In Your Own Words

How does this mini search engine differ from a keyword search (like grep)? What are the advantages and limitations? Write your answer here.

## Similarity Matrix

In [ ]:
# Compare all documents against each other
sim_matrix = cosine_similarity(doc_embeddings)

# Print as a readable table (first 5 docs for readability)
import pandas as pd

labels = [f"doc_{i}" for i in range(5)]
df = pd.DataFrame(
    sim_matrix[:5, :5].round(2),
    index=labels,
    columns=labels,
)
print("Similarity matrix (first 5 documents):")
print(df)
print("\nDiagonal is always 1.0 (document vs itself)")
print("High off-diagonal values indicate similar documents")

In [ ]:
# Experiment: find the most similar pair of documents
max_sim = 0
max_pair = (0, 0)
for i in range(len(documents)):
    for j in range(i + 1, len(documents)):
        if sim_matrix[i][j] > max_sim:
            max_sim = sim_matrix[i][j]
            max_pair = (i, j)

i, j = max_pair
print(f"Most similar pair (score: {max_sim:.3f}):")
print(f"  '{documents[i]}'")
print(f"  '{documents[j]}'")

### ✍️ In Your Own Words

How could a similarity matrix be useful in a real application? Think about clustering, deduplication, or recommendation systems. Write your answer here.

## Recap

- ✅ Cosine similarity measures angle between vectors (−1 to 1)
- ✅ Your manual implementation matches sklearn's — it's just dot product / magnitudes
- ✅ Semantic similarity ≠ lexical similarity — embeddings capture meaning
- ✅ Models struggle with negation — a known limitation
- ✅ Semantic search: embed query → compare to document embeddings → rank by similarity
- ✅ This is literally what vector databases do — you just built the core algorithm
- ✅ Similarity matrices reveal document clusters and relationships

**Next:** [02_ner_and_classification.ipynb](./02_ner_and_classification.ipynb) — extract structure from text